In [50]:
from gtda.homology import VietorisRipsPersistence, CubicalPersistence
from gtda.diagrams import PersistenceEntropy, Scaler, PersistenceLandscape
from gtda.plotting import plot_heatmap, plot_point_cloud, plot_diagram
import numpy as np
from tqdm import tqdm
import pandas as pd

In [51]:
#def function for calculation features
def barcode_entropy_1(matrix, dim=0):
    lengths = []
    for barcode in matrix:
      if barcode[2]==dim:
        lengths.append(barcode[1]-barcode[0])
    lengths /= np.sum(lengths)
    return -np.sum(lengths*np.log(lengths))

def barcode_sum_1(matrix, dim=0):
    """Calculate sum of lengths of barcodes in h{dim}"""
    lengths = []
    for barcode in matrix:
      if len(barcode):
        if barcode[2]==dim:
          lengths.append(barcode[1] - barcode[0])
      else:
        return 0.0
    return np.sum(lengths)


def barcode_mean_1(matrix, dim=0):
    """Calculate mean of lengths of barcodes in h{dim}"""
    lengths = []
    for barcode in matrix:
      if len(barcode):
        if barcode[2]==dim:
          lengths.append(barcode[1] - barcode[0])
      else:
        return 0.0
    return np.mean(lengths)

def barcode_std_1(matrix, dim=0):
    """Calculate std of lengths of barcodes in h{dim}"""
    lengths = []
    for barcode in matrix:
      if len(barcode):
        if barcode[2]==dim:
          lengths.append(barcode[1] - barcode[0])
      else:
          return 0.0
    return np.std(lengths)

In [52]:
#def function for prepare vector of topologies feature
def calculate_tda_feature(diagrams_basic):
        sum_at_0 = []
        sum_at_1 = []
        sum_at_2 = []
        mean_at_0 = []
        mean_at_1 = []
        mean_at_2 = []
        std_at_0 = []
        std_at_1 = []
        std_at_2 = []
        entropy_at_0 = []
        entropy_at_1 = []
        entropy_at_2 = []
        for matrix in diagrams_basic:
            sum_at_0.append(barcode_sum_1(matrix, 0))
            sum_at_1.append(barcode_sum_1(matrix, 1))
            sum_at_2.append(barcode_sum_1(matrix, 2))
            mean_at_0.append(barcode_mean_1(matrix, 0))
            mean_at_1.append(barcode_mean_1(matrix, 1))
            mean_at_2.append(barcode_mean_1(matrix, 2))
            std_at_0.append(barcode_std_1(matrix, 0))
            std_at_1.append(barcode_std_1(matrix, 1))
            std_at_2.append(barcode_std_1(matrix, 2))
            entropy_at_0.append(barcode_entropy_1(matrix, 0))
            entropy_at_1.append(barcode_entropy_1(matrix, 1))
            entropy_at_2.append(barcode_entropy_1(matrix, 2))
        sum_vector = np.array([np.sum([sum_at_0,sum_at_1,sum_at_2]),np.sum([mean_at_0,mean_at_1,mean_at_2]), np.sum([std_at_0,std_at_1,std_at_2]), np.sum([entropy_at_0,entropy_at_1,entropy_at_2])])
        concat_vector = np.array([sum_at_0, sum_at_1,sum_at_2, mean_at_0, mean_at_1,mean_at_2, std_at_0, std_at_1,std_at_2, entropy_at_0, entropy_at_1,entropy_at_2]).flatten()
        return sum_vector, concat_vector

In [53]:
!mkdir Topology/XYZ_persistence_barcodes_metal

In [54]:
folder_1='./optimized_complexes_rdkit'
try_count = 0
except_count = 0
check_files=[]
import os
import pickle
for filename in tqdm(os.listdir(folder_1)):
    print(f'Work with Metal {filename}')
    disk_loc = filename.split('.')[0]
    name = disk_loc
    file_location = f'{folder_1}/{filename}'
    xyz_file = np.genfromtxt(fname=file_location, skip_header=2, dtype='unicode')
    coordinates = (xyz_file[:,1:])
    coordinates = coordinates.astype(np.float16)
    persistence = VietorisRipsPersistence(metric="euclidean", homology_dimensions=[0,1,2], n_jobs=6)
    temp_new = coordinates.reshape(1, *coordinates.shape)
    diagrams_basic = persistence.fit_transform(temp_new)
    with open(f"./Topology/XYZ_persistence_barcodes_metal/diagrams_basic_{name}.pkl", "wb") as f:
      pickle.dump(diagrams_basic, f)
    try_count+=1
print((try_count,except_count))

Work with Metal complex_3452.xyz
Work with Metal complex_2994.xyz
Work with Metal complex_13690.xyz
Work with Metal complex_18935.xyz
Work with Metal complex_2764.xyz
Work with Metal complex_4315.xyz
Work with Metal complex_14869.xyz
Work with Metal complex_9157.xyz
Work with Metal complex_6264.xyz
Work with Metal complex_11911.xyz
Work with Metal complex_979.xyz
Work with Metal complex_9143.xyz
Work with Metal complex_6270.xyz
Work with Metal complex_11905.xyz


100%|██████████| 10173/10173 [04:00<00:00, 42.34it/s]

Work with Metal complex_4467.xyz
Work with Metal complex_5779.xyz
Work with Metal complex_7608.xyz
Work with Metal complex_1279.xyz
Work with Metal complex_9625.xyz
Work with Metal complex_18921.xyz
(10173, 0)


In [55]:
!mkdir Topology/topology_features_metal

In [ ]:
import os
import pickle
list_of_feature = []
folder = './Topology/XYZ_persistence_barcodes_metal'
for filename in tqdm(os.listdir(folder)):
    with open(f"{folder}/{filename}", "rb") as f:
      diagrams_basic = pickle.load(f)
    local_conc = []
    local_sum = []
    name = filename.split('.')[0]
    print(name)
    sum_v, concat_v = calculate_tda_feature(diagrams_basic)
    local_conc.append(concat_v)
    local_sum.append(sum_v)
    print('Sum_and_conc is OK!')
    
    df_sum = pd.DataFrame(np.array(local_sum))
    df_sum.to_csv(f'./Topology/topology_features_metal/{name}_sum.csv')
    df_conc = pd.DataFrame(np.array(local_conc))
    df_conc.to_csv(f'./Topology/topology_features_metal/{name}_conc.csv')
    list_of_feature.append(np.array(local_conc))
    print('DF is LOAD!')

# Solvents

In [57]:
!mkdir Topology/XYZ_persistence_barcodes_metal_solvent
!mkdir Topology/topology_features_metal_solvent

In [58]:
folder_1='./optimized_solvents'
try_count = 0
except_count = 0
check_files=[]
error=[]
import os
import pickle
for filename in tqdm(os.listdir(folder_1)):
  try:
    disk_loc = filename.split('.')[0]
    print(f'Work with Solvent {disk_loc.lower()}')
    file_location = f'{folder_1}/{filename}'
    xyz_file = np.genfromtxt(fname=file_location, skip_header=2, dtype='unicode')
    coordinates = (xyz_file[:,1:])
    coordinates = coordinates.astype(np.float16)
    persistence = VietorisRipsPersistence(metric="euclidean", homology_dimensions=[0,1,2], n_jobs=6)
    temp_new = coordinates.reshape(1, *coordinates.shape)
    diagrams_basic = persistence.fit_transform(temp_new)
    with open(f"./Topology/XYZ_persistence_barcodes_metal_solvent/diagrams_basic_{disk_loc.lower()}.pkl", "wb") as f:
      pickle.dump(diagrams_basic, f)
    try_count+=1
  except:
    print(f'NOT WORK with Solvent {filename}')
    error.append(filename)

print((try_count,except_count))

  0%|          | 0/194 [00:00<?, ?it/s]/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  check_point_clouds(X, accept_sparse=True,
/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:299: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  X = check_point_clouds(X, accept_sparse=True,
  4%|▍         | 8/194 [00:00<00:02, 74.78it/s]/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input i

Work with Solvent acetonitrile_aq_phosphate_buffer
Work with Solvent benzene
Work with Solvent heptane
Work with Solvent 1_2-dichloro-ethane
Work with Solvent dichloromethane_triethylamine
Work with Solvent propan-2-ol
Work with Solvent isopropyl_alcohol
Work with Solvent hydrogenchloride_water_tetrahydrofuran
Work with Solvent water_monomer_tert-butyl_alcohol
Work with Solvent aq_koh
Work with Solvent chcl3
Work with Solvent chlorobenzene
Work with Solvent aq_buffer_water_acetonitrile
Work with Solvent cdcl3
Work with Solvent dichlorofluoromethane


  8%|▊         | 16/194 [00:00<00:02, 74.47it/s]/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  check_point_clouds(X, accept_sparse=True,
/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:299: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  X = check_point_clouds(X, accept_sparse=True,
/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vec

Work with Solvent methyl_cyclohexane_trichloroethylene
Work with Solvent diethyl_ether_methylbutane_ethanol
Work with Solvent water_monomer_dimethyl_sulfoxide
Work with Solvent aq_hf
Work with Solvent acidic_aq_solution_methanol
Work with Solvent aq_ammonia_nh3
Work with Solvent hydrogenchloride
Work with Solvent toluene
Work with Solvent chloroform_methanol
Work with Solvent n_n-dimethyl-formamide_trifluoroacetic_acid
Work with Solvent octanol
Work with Solvent water_methanol
Work with Solvent ethyl_acetate
Work with Solvent methyl_cyclohexane
Work with Solvent aq_naoh
Work with Solvent m-xylene


/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  check_point_clouds(X, accept_sparse=True,
/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:299: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  X = check_point_clouds(X, accept_sparse=True,
 16%|█▋        | 32/194 [00:00<00:02, 74.24it/s]/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vec

...

Work with Solvent 2-methyltetrahydrofuran
Work with Solvent methanol_acetic_acid
Work with Solvent 2_h6_acetone
Work with Solvent n-heptane
Work with Solvent aq_phosphate_buffer_acetonitrile
Work with Solvent tetrahydrofuran_benzene
Work with Solvent formamide
Work with Solvent chloroform_ethanol_acetic_acid
Work with Solvent dichloromethane-d2
Work with Solvent acetonitrile_acetic_acid
Work with Solvent nitrobenzene
Work with Solvent n_n-dimethyl-formamide_dichloromethane
Work with Solvent aq_buffer_dimethyl_sulfoxide
Work with Solvent freon_113_1_1_2-trichloro-trifluoroethane
Work with Solvent dichloromethane_methanol


/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  check_point_clouds(X, accept_sparse=True,
/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:299: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  X = check_point_clouds(X, accept_sparse=True,
/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  check_point_clouds(X,

Work with Solvent n_n-dimethyl-formamide_water_monomer
Work with Solvent aq_h2so4
Work with Solvent hydrogenchloride_water_monomer
Work with Solvent dichloromethane_dimethyl_sulfoxide
Work with Solvent 1_4-dioxane
Work with Solvent water_hydrogen_cyanide
Work with Solvent water-d2
Work with Solvent aq_buffer_dimethyl_sulfoxide_sodium_chloride
Work with Solvent dichloromethane
Work with Solvent 2_2_2-trifluoroethanol_water_trifluorormethanesulf
Work with Solvent dichloroethane_1_2-dichloroethane
Work with Solvent water_n_n-dimethyl-formamide
Work with Solvent 2_h8-toluene
Work with Solvent tetrachloromethane
Work with Solvent aq_buffer_n_n-dimethyl-formamide
Work with Solvent aq_acetate_buffer_dimethyl_sulfoxide


 95%|█████████▍| 184/194 [00:02<00:00, 74.77it/s]/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  check_point_clouds(X, accept_sparse=True,
/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:299: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but the input is being treated as a collection of vectors in Euclidean space.
  X = check_point_clouds(X, accept_sparse=True,
 99%|█████████▉| 192/194 [00:02<00:00, 74.55it/s]/opt/anaconda3/lib/python3.12/site-packages/gtda/homology/simplicial.py:232: DataDimensionalityWarning: Input array X has X.shape[1] == X.shape[2]. This is consistent with a collection of distance/adjacency matrices, but

Work with Solvent isooctane
Work with Solvent dimethyl_sulfoxide
Work with Solvent ethanol
Work with Solvent water_dimethyl_sulfoxide
Work with Solvent water_tert-butyl_alcohol
Work with Solvent glycerol
Work with Solvent 1_2-dimethoxyethane
Work with Solvent tert-butyl_alcohol
Work with Solvent butan-1-ol
Work with Solvent dimethyl_sulfoxide_n_n-dimethyl-formamide
Work with Solvent ethanol_water
Work with Solvent cs2
Work with Solvent n_n-dimethyl-formamide
(194, 0)


In [59]:
import os
import pickle
list_of_feature = []
folder = './Topology/XYZ_persistence_barcodes_metal_solvent'
for filename in tqdm(os.listdir(folder)):
    try:
      with open(f"{folder}/{filename}", "rb") as f:
        diagrams_basic = pickle.load(f)
      local_conc = []
      local_sum = []
      name = filename.split('.')[0]
      print(name)
      sum_v, concat_v = calculate_tda_feature(diagrams_basic)
      local_conc.append(concat_v)
      local_sum.append(sum_v)
      # print('Sum_and_conc is OK!')
      df_sum = pd.DataFrame(np.array(local_sum))
      df_sum.to_csv(f'./Topology/topology_features_metal_solvent/{name}_sum.csv')
      df_conc = pd.DataFrame(np.array(local_conc))
      df_conc.to_csv(f'./Topology/topology_features_metal_solvent/{name}_conc.csv')
      list_of_feature.append(np.array(local_conc))
      # print('DF is LOAD!')
    except:
      print(f'Not work with {name}')
      df_sum = pd.DataFrame(np.array([0.0 for i in range(4)]))
      df_sum.to_csv(f'./Topology/topology_features_metal_solvent/{name}_sum.csv')
      df_conc = pd.DataFrame(np.array([0.0 for i in range(4)]))
      df_conc.to_csv(f'./Topology/topology_features_metal_solvent/{name}_conc.csv')
      # list_of_feature.append(np.array(local_conc))

  0%|          | 0/194 [00:00<?, ?it/s]/var/folders/df/zxk1r7d94k9c_444dg29h3wh0000gn/T/ipykernel_90543/2425355246.py:7: RuntimeWarning: invalid value encountered in divide
  lengths /= np.sum(lengths)
100%|██████████| 194/194 [00:00<00:00, 1230.47it/s]

diagrams_basic_benzene
diagrams_basic_heptane
diagrams_basic_cyclohexane
diagrams_basic_acidic_aq_solution
diagrams_basic_2_2_2-trifluoroethanol_water_trifluorormethanesulf
diagrams_basic_acetonitrile
diagrams_basic_dimethyl_sulfoxide_aq_phosphate_buffer
diagrams_basic_ethyl_acetate
diagrams_basic_ethanol_water_monomer
diagrams_basic_chloroform_triethylamine
diagrams_basic_nitromethane
diagrams_basic_m-xylene
diagrams_basic_methanol_aq_phosphate_buffer
diagrams_basic_acetone_hexane
diagrams_basic_methanol_water
diagrams_basic_chloroform_acetone
diagrams_basic_methanol_n_n-dimethyl-formamide
diagrams_basic_water_ethanol
diagrams_basic_n_n-dimethyl-formamide_water_monomer
diagrams_basic_propan-1-ol
diagrams_basic_liquid_carbon_dioxide
diagrams_basic_methanol
diagrams_basic_acetonitrile_methanol
diagrams_basic_water_tetrahydrofuran
diagrams_basic_dimethylsulfoxide-d6
diagrams_basic_ch2cl2
diagrams_basic_hexane
diagrams_basic_acetone_water_monomer
diagrams_basic_aq_buffer_acetonitrile
diag

In [60]:
df_conc

,0,1,2,3,4,5,6,7,8,9,10,11
0,14.634249,0.834903,0.0,1.219521,0.834903,0.0,0.182091,0.0,0.0,2.474138,-0.0,NaN


In [61]:
### Load metal target dataset
start_metal = pd.read_csv('/Users/egorilin/Desktop/MSU_AI/processed_complexes_2.csv', sep=';') # load dataset with metal complex targets
start_metal
topology_full = {}


In [62]:
start_metal

,Metal_Complex_SMILES,Reaxys_Registry_Number,Solvent_Name,Solvent_SMILES,Selected_Absorption,lig_1,lig_2,lig_3,lig_4,lig_5
0,C[C-]12[Ir+]3456([Cl][Ir+]789%10([Cl]3)([C-]3(...,12591084,ethanol,CCO,NaN,C[c-]1c(C)c(C)c(c1C)C,[Cl],[Cl],[c-]1(C)c(C)c(C)c(c1C)C,[Cl]
1,C[C-]12[Ir+]3456([Cl][Ir+]789%10([Cl]3)([C-]3(...,12591084,dichloromethane,ClCCl,NaN,C[c-]1c(C)c(C)c(c1C)C,[Cl],[Cl],[c-]1(C)c(C)c(C)c(c1C)C,[Cl]
2,C[C-]12[Ir+]3456([Cl][Ir+]789%10([Cl]3)([C-]3(...,12591084,ethanol,CCO,NaN,C[c-]1c(C)c(C)c(c1C)C,[Cl],[Cl],[c-]1(C)c(C)c(C)c(c1C)C,[Cl]
3,C[C-]12[Ir+]3456([Cl][Ir+]789%10([Cl]3)([C-]3(...,12591084,NaN,NaN,417.0,C[c-]1c(C)c(C)c(c1C)C,[Cl],[Cl],[c-]1(C)c(C)c(C)c(c1C)C,[Cl]
4,[H][C]12=[C]3([H])CC[C]4([H])=[C]([H])(CC1)[Ir...,11993489,NaN,NaN,NaN,C1=CCCC=CCC1,[Cl],[Cl],C1=CCCC=CCC1,NaN
...,...,...,...,...,...,...,...,...,...,...
19163,CCCCN1C=CN2[C]1[Ru++]1345[C]6N(CCCC)C=CN6C6=CC...,17791074,acetonitrile,CC#N,385.0,CCCCN1C=CN([C]1)c1nc(N2C=CN(CCCC)[C]2)ccc1,[C]1N(CCCC)C=CN1c1cccc(N2C=CN(CCCC)[C]2)n1,NaN,NaN,NaN
19164,C1C2=CC=CC=[N]2[Ru++]2345[N]6=CC=CC=C6C6=CC=CC...,17791345,dichloromethane,ClCCl,370.0,C(c1ccccn1)N(Cc1ncccc1)Cc1ncccc1,n1ccccc1c1ccccn1,NaN,NaN,NaN
19165,[O-]C(=O)C1=CC2=[N]3C(=C1)C1=[N](C=CC=C1)[Ru++...,18046639,water; methanol,CO,485.0,[O-]C(=O)c1cc(nc(c1)c1ncccc1)c1ncccc1,n1c(cccc1)c1cc(cc(c2ncccc2)n1)C(=O)[O-],NaN,NaN,NaN
19166,CC1=CC2=[N](C=C1)[Ru++]([Cl-])([Cl-])(C#[O])(C...,18047831,acetonitrile,CC#N,NaN,Cc1cc(ncc1)c1nccc(c1)C(=O)O,[Cl-],[Cl-],[C]#[O],[C]#[O]


In [77]:
i = 7
solv = start_metal['Solvent_Name'][0]
solv_top = pd.read_csv(f'./Topology/topology_features_metal_solvent/diagrams_basic_{solv.lower()}_conc.csv')
metal_top = pd.read_csv(f'./Topology/topology_features_metal/diagrams_basic_complex_{i}_conc.csv')
full = pd.concat([metal_top, solv_top], axis=1).drop(['Unnamed: 0'], axis=1)
topology_full[f'{i}'] = np.array(full).squeeze()

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import re

# Инициализируем словарь для топологических фичей
topology_full = {}

# Определяем количество фичей для металла и растворителя
n_metal_features = 12  # количество столбцов в columns_metal
n_solv_features = 12   # количество столбцов в columns_solv
total_features = n_metal_features + n_solv_features

# Функция для очистки названия растворителя для имени файла
def sanitize_solvent_name(solv_name):
    """
    Очищает название растворителя для использования в имени файла:
    - Заменяет ';' на '_'
    - Заменяет другие недопустимые символы
    - Приводит к нижнему регистру
    """
    if not isinstance(solv_name, str):
        return str(solv_name).lower()
    
    # Заменяем ';' на '_'
    cleaned = solv_name.replace(';', '_')
    
    # Заменяем другие потенциально проблемные символы
    cleaned = re.sub(r'[\\/*?:"<>|]', '_', cleaned)
    
    # Заменяем пробелы на подчеркивания
    cleaned = cleaned.replace(' ', '_')
    cleaned = cleaned.replace(',', '_')
    
    # Приводим к нижнему регистру
    cleaned = cleaned.lower()
    
    return cleaned

# Сначала собираем все данные с улучшенной обработкой ошибок
for i in tqdm(range(len(start_metal))):
    try:
        # Пытаемся прочитать данные по металлу
        try:
            metal_path = f'./Topology/topology_features_metal/diagrams_basic_complex_{i}_conc.csv'
            if os.path.exists(metal_path):
                metal_top = pd.read_csv(metal_path)
                metal_data = metal_top.drop(['Unnamed: 0'], axis=1).values.squeeze()
                metal_success = True
                print(f"✅ Metal data loaded successfully for index {i}")
            else:
                print(f"❌ Metal file not found for index {i}: {metal_path}")
                metal_success = False
                metal_data = np.full(n_metal_features, np.nan)
        except Exception as metal_e:
            print(f"❌ Metal data failed for index {i}: {metal_e}")
            metal_success = False
            metal_data = np.full(n_metal_features, np.nan)
        
        # Пытаемся прочитать данные по растворителю
        try:
            solv_original = start_metal['Solvent_Name'][i]
            solv_clean = sanitize_solvent_name(solv_original)
            
            # Формируем путь к файлу растворителя
            solv_path = f'./Topology/topology_features_metal_solvent/diagrams_basic_{solv_clean}_conc.csv'
            
            # Дополнительная проверка: пробуем разные варианты имени файла
            solv_data = None
            solv_success = False
            
            # Основной путь
            if os.path.exists(solv_path):
                solv_top = pd.read_csv(solv_path)
                solv_data = solv_top.drop(['Unnamed: 0'], axis=1).values.squeeze()
                solv_success = True
                print(f"✅ Solvent data loaded successfully for index {i} (original: '{solv_original}', cleaned: '{solv_clean}')")
            else:
                # Пробуем альтернативные варианты именования
                alternative_names = [
                    solv_clean.replace('_', ''),  # убираем все подчеркивания
                    solv_clean.replace(';', '_'),
                    solv_clean.replace(' ', '_'),
                    solv_clean.lower(),
                    solv_clean.upper(),
                ]
                
                for alt_name in alternative_names:
                    alt_path = f'./Topology/topology_features_metal_solvent/diagrams_basic_{alt_name}_conc.csv'
                    if os.path.exists(alt_path):
                        print(f"🔄 Found solvent file with alternative name for index {i}: '{alt_name}'")
                        solv_top = pd.read_csv(alt_path)
                        solv_data = solv_top.drop(['Unnamed: 0'], axis=1).values.squeeze()
                        solv_success = True
                        break
                
                if not solv_success:
                    print(f"❌ Solvent file not found for index {i} after trying all alternatives. Original: '{solv_original}', Cleaned: '{solv_clean}'")
                    solv_data = np.full(n_solv_features, np.nan)
                    
        except Exception as solv_e:
            print(f"❌ Solvent data failed for index {i}: {solv_e}")
            solv_success = False
            solv_data = np.full(n_solv_features, np.nan)
        
        # Формируем полный вектор фичей
        if metal_success and solv_success:
            # Оба набора данных успешно загружены
            full_data = np.concatenate([metal_data, solv_data])
            topology_full[f'{i}'] = full_data
            print(f"🌟 Both metal and solvent data combined for index {i}")
        elif metal_success:
            # Только данные по металлу доступны
            full_data = np.concatenate([metal_data, np.full(n_solv_features, np.nan)])
            topology_full[f'{i}'] = full_data
            print(f"⚠️ Only metal data available for index {i}, solvent features set to NaN")
        elif solv_success:
            # Только данные по растворителю доступны
            full_data = np.concatenate([np.full(n_metal_features, np.nan), solv_data])
            topology_full[f'{i}'] = full_data
            print(f"⚠️ Only solvent data available for index {i}, metal features set to NaN")
        else:
            # Оба набора данных недоступны
            topology_full[f'{i}'] = np.full(total_features, np.nan)
            print(f"❌ No data available for index {i}")
            
    except Exception as e:
        print(f"🔥 Critical error for index {i}: {e}")
        topology_full[f'{i}'] = np.full(total_features, np.nan)
    
    print("-" * 50)

# Создаем DataFrame
columns_metal = ['sum_H_0', 'sum_H_1', 'sum_H_2', 'mean_H_0', 'mean_H_1', 'mean_H_2', 
                'std_H_0', 'std_H_1', 'std_H_2', 'entropy_H_0', 'entropy_H_1', 'entropy_H_2']
columns_solv = ['solv_sum_H_0', 'solv_sum_H_1', 'solv_sum_H_2', 'solv_mean_H_0', 'solv_mean_H_1', 'solv_mean_H_2', 
                'solv_std_H_0', 'solv_std_H_1', 'solv_std_H_2', 'solv_entropy_H_0', 'solv_entropy_H_1', 'solv_entropy_H_2']
full_topol = columns_metal + columns_solv

topol_feat = pd.DataFrame.from_dict(topology_full, orient='index', columns=full_topol)

# Проверяем форму данных перед импутацией
print(f"\n📊 Shape of topol_feat: {topol_feat.shape}")
print(f"🔢 Number of NaN values before imputation: {topol_feat.isna().sum().sum()}")
print("\n🗂️ NaN counts per column before imputation:")
print(topol_feat.isna().sum())

# Двухэтапная импутация:
# Шаг 1: Заполнение методом 'nearest' (ближайшее значение)
topol_feat_nearest = topol_feat.copy()

# Применяем interpolate с методом 'nearest' для каждого столбца
for column in topol_feat_nearest.columns:
    # Проверяем, есть ли вообще непустые значения в столбце
    if topol_feat_nearest[column].notna().any():
        # Заполняем пропуски ближайшими значениями
        topol_feat_nearest[column] = topol_feat_nearest[column].interpolate(method='nearest', limit_direction='both')
        
        # Если остались пропуски, заполняем их ближайшим доступным значением
        if topol_feat_nearest[column].isna().any():
            # Заполняем оставшиеся NaN ближайшим непустым значением
            topol_feat_nearest[column] = topol_feat_nearest[column].fillna(method='ffill')
            topol_feat_nearest[column] = topol_feat_nearest[column].fillna(method='bfill')
    else:
        print(f"⚠️ Column {column} has no valid values. Filling with zeros for now.")
        topol_feat_nearest[column] = 0

# Шаг 2: Заполнение оставшихся пропусков медианой
topol_feat_final = topol_feat_nearest.copy()
for column in topol_feat_final.columns:
    # Проверяем, есть ли непустые значения для вычисления медианы
    if topol_feat_final[column].notna().any():
        median_value = topol_feat_final[column].median()
        topol_feat_final[column] = topol_feat_final[column].fillna(median_value)
        print(f"🔧 Column {column}: filled remaining NaNs with median={median_value:.4f}")
    else:
        # Если в столбце нет непустых значений, заполняем нулями
        topol_feat_final[column] = 0
        print(f"🔧 Column {column} had no valid values. Filled with zeros.")

# Проверяем, что все пропуски заполнены
remaining_nans = topol_feat_final.isna().sum().sum()
if remaining_nans > 0:
    print(f"🚨 Warning: {remaining_nans} NaN values remain after imputation. Filling with column medians...")
    for column in topol_feat_final.columns:
        if topol_feat_final[column].isna().any():
            median_value = topol_feat_final[column].median() if topol_feat_final[column].notna().any() else 0
            topol_feat_final[column] = topol_feat_final[column].fillna(median_value)

# Проверка финального результата
print("\n" + "="*60)
print("🎯 FINAL RESULTS:")
print(f"📏 Shape of final DataFrame: {topol_feat_final.shape}")
print(f"❌ Total remaining NaN values: {topol_feat_final.isna().sum().sum()}")
print("\n🗂️ NaN counts per column after final imputation:")
print(topol_feat_final.isna().sum())

# Сохраняем результат
topol_feat_final.to_csv('./Topology/Topology_features_metal_complex_imputed.csv')

# Также сохраняем исходный DataFrame с пропусками для отладки
topol_feat.to_csv('./Topology/Topology_features_metal_complex_with_nan.csv')

# Создаем отчет о заполнении данных
report_data = []
for i in range(len(start_metal)):
    row = topol_feat.iloc[i]
    metal_features_available = row[columns_metal].notna().all()
    solv_features_available = row[columns_solv].notna().all()
    
    # Проверяем, какие именно фичи доступны
    metal_available_count = row[columns_metal].notna().sum()
    solv_available_count = row[columns_solv].notna().sum()
    
    report_data.append({
        'index': i,
        'metal_available': metal_features_available,
        'solv_available': solv_features_available,
        'metal_available_count': metal_available_count,
        'solv_available_count': solv_available_count,
        'total_available': row.notna().sum(),
        'solvent_original': start_metal['Solvent_Name'][i] if i < len(start_metal) else None
    })

report_df = pd.DataFrame(report_data)
report_df.to_csv('./Topology/data_availability_report.csv', index=False)

print("\n📋 Data availability report saved to './Topology/data_availability_report.csv'")
print(f"📈 Summary: {report_df['metal_available'].sum()} metal datasets fully available, {report_df['solv_available'].sum()} solvent datasets fully available")
print(f"📊 Total metal features available: {report_df['metal_available_count'].sum()}/{len(report_df)*n_metal_features}")
print(f"📊 Total solvent features available: {report_df['solv_available_count'].sum()}/{len(report_df)*n_solv_features}")

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import re
import warnings

# Инициализируем словарь для топологических фичей
topology_full = {}

# Определяем количество фичей для металла и растворителя
n_metal_features = 12  # количество столбцов в columns_metal
n_solv_features = 12   # количество столбцов в columns_solv
total_features = n_metal_features + n_solv_features

# Функция для очистки названия растворителя для имени файла
def sanitize_solvent_name(solv_name):
    """
    Очищает название растворителя для использования в имени файла:
    - Заменяет ';' на '_'
    - Заменяет другие недопустимые символы
    - Приводит к нижнему регистру
    """
    if not isinstance(solv_name, str):
        return str(solv_name).lower()
    
    # Заменяем ';' на '_'
    cleaned = solv_name.replace(';', '_')
    
    # Заменяем другие потенциально проблемные символы
    cleaned = re.sub(r'[\\/*?:"<>|]', '_', cleaned)
    
    # Заменяем пробелы на подчеркивания
    cleaned = cleaned.replace(' ', '_')
    cleaned = cleaned.replace(',', '_')
    
    # Приводим к нижнему регистру
    cleaned = cleaned.lower()
    
    return cleaned

# Сначала собираем все данные с улучшенной обработкой ошибок
for i in tqdm(range(len(start_metal))):
    try:
        # Пытаемся прочитать данные по металлу
        try:
            metal_path = f'./Topology/topology_features_metal/diagrams_basic_complex_{i}_conc.csv'
            if os.path.exists(metal_path):
                metal_top = pd.read_csv(metal_path)
                # Удаляем столбец с индексом если он есть
                if 'Unnamed: 0' in metal_top.columns:
                    metal_top = metal_top.drop(['Unnamed: 0'], axis=1)
                # Преобразуем все значения в числовые
                metal_top = metal_top.apply(pd.to_numeric, errors='coerce')
                metal_data = metal_top.values.squeeze()
                metal_success = True
                if metal_data.shape[0] != n_metal_features:
                    print(f"⚠️ Warning: Metal data for index {i} has {metal_data.shape[0]} features instead of {n_metal_features}")
                    metal_success = False
                    metal_data = np.full(n_metal_features, np.nan)
                print(f"✅ Metal data loaded successfully for index {i}")
            else:
                print(f"❌ Metal file not found for index {i}: {metal_path}")
                metal_success = False
                metal_data = np.full(n_metal_features, np.nan)
        except Exception as metal_e:
            print(f"❌ Metal data failed for index {i}: {metal_e}")
            metal_success = False
            metal_data = np.full(n_metal_features, np.nan)
        
        # Пытаемся прочитать данные по растворителю
        try:
            solv_original = start_metal['Solvent_Name'][i]
            solv_clean = sanitize_solvent_name(solv_original)
            
            # Формируем путь к файлу растворителя
            solv_path = f'./Topology/topology_features_metal_solvent/diagrams_basic_{solv_clean}_conc.csv'
            
            # Дополнительная проверка: пробуем разные варианты имени файла
            solv_data = None
            solv_success = False
            
            # Основной путь
            if os.path.exists(solv_path):
                solv_top = pd.read_csv(solv_path)
                if 'Unnamed: 0' in solv_top.columns:
                    solv_top = solv_top.drop(['Unnamed: 0'], axis=1)
                # Преобразуем все значения в числовые
                solv_top = solv_top.apply(pd.to_numeric, errors='coerce')
                solv_data = solv_top.values.squeeze()
                if solv_data.shape[0] != n_solv_features:
                    print(f"⚠️ Warning: Solvent data for index {i} has {solv_data.shape[0]} features instead of {n_solv_features}")
                    solv_success = False
                    solv_data = np.full(n_solv_features, np.nan)
                else:
                    solv_success = True
                print(f"✅ Solvent data loaded successfully for index {i} (original: '{solv_original}', cleaned: '{solv_clean}')")
            else:
                # Пробуем альтернативные варианты именования
                alternative_names = [
                    solv_clean.replace('_', ''),  # убираем все подчеркивания
                    solv_clean.replace(';', '_'),
                    solv_clean.replace(' ', '_'),
                    solv_clean.lower(),
                    solv_clean.upper(),
                    solv_clean.replace('-', '_'),
                    solv_clean.replace('.', '_'),
                ]
                
                for alt_name in alternative_names:
                    alt_path = f'./Topology/topology_features_metal_solvent/diagrams_basic_{alt_name}_conc.csv'
                    if os.path.exists(alt_path):
                        print(f"🔄 Found solvent file with alternative name for index {i}: '{alt_name}'")
                        solv_top = pd.read_csv(alt_path)
                        if 'Unnamed: 0' in solv_top.columns:
                            solv_top = solv_top.drop(['Unnamed: 0'], axis=1)
                        # Преобразуем все значения в числовые
                        solv_top = solv_top.apply(pd.to_numeric, errors='coerce')
                        solv_data = solv_top.values.squeeze()
                        if solv_data.shape[0] != n_solv_features:
                            print(f"⚠️ Warning: Alternative solvent data for index {i} has {solv_data.shape[0]} features instead of {n_solv_features}")
                            solv_success = False
                            solv_data = np.full(n_solv_features, np.nan)
                        else:
                            solv_success = True
                        break
                
                if not solv_success:
                    print(f"❌ Solvent file not found for index {i} after trying all alternatives. Original: '{solv_original}', Cleaned: '{solv_clean}'")
                    solv_data = np.full(n_solv_features, np.nan)
                    
        except Exception as solv_e:
            print(f"❌ Solvent data failed for index {i}: {solv_e}")
            solv_success = False
            solv_data = np.full(n_solv_features, np.nan)
        
        # Формируем полный вектор фичей
        if metal_success and solv_success:
            # Оба набора данных успешно загружены
            try:
                full_data = np.concatenate([metal_data, solv_data])
                if full_data.shape[0] == total_features:
                    topology_full[f'{i}'] = full_data
                    print(f"🌟 Both metal and solvent data combined for index {i}")
                else:
                    print(f"❌ Concatenation failed for index {i}: unexpected shape {full_data.shape}")
                    topology_full[f'{i}'] = np.full(total_features, np.nan)
            except Exception as concat_e:
                print(f"❌ Concatenation error for index {i}: {concat_e}")
                topology_full[f'{i}'] = np.full(total_features, np.nan)
        elif metal_success:
            # Только данные по металлу доступны
            full_data = np.concatenate([metal_data, np.full(n_solv_features, np.nan)])
            topology_full[f'{i}'] = full_data
            print(f"⚠️ Only metal data available for index {i}, solvent features set to NaN")
        elif solv_success:
            # Только данные по растворителю доступны
            full_data = np.concatenate([np.full(n_metal_features, np.nan), solv_data])
            topology_full[f'{i}'] = full_data
            print(f"⚠️ Only solvent data available for index {i}, metal features set to NaN")
        else:
            # Оба набора данных недоступны
            topology_full[f'{i}'] = np.full(total_features, np.nan)
            print(f"❌ No data available for index {i}")
            
    except Exception as e:
        print(f"🔥 Critical error for index {i}: {e}")
        topology_full[f'{i}'] = np.full(total_features, np.nan)
    
    print("-" * 50)


In [97]:
# Создаем DataFrame
columns_metal = ['sum_H_0', 'sum_H_1', 'sum_H_2', 'mean_H_0', 'mean_H_1', 'mean_H_2', 
                'std_H_0', 'std_H_1', 'std_H_2', 'entropy_H_0', 'entropy_H_1', 'entropy_H_2']
columns_solv = ['solv_sum_H_0', 'solv_sum_H_1', 'solv_sum_H_2', 'solv_mean_H_0', 'solv_mean_H_1', 'solv_mean_H_2', 
                'solv_std_H_0', 'solv_std_H_1', 'solv_std_H_2', 'solv_entropy_H_0', 'solv_entropy_H_1', 'solv_entropy_H_2']
full_topol = columns_metal + columns_solv

topol_feat = pd.DataFrame.from_dict(topology_full, orient='index', columns=full_topol)

# Проверяем форму данных перед импутацией
print(f"\n📊 Shape of topol_feat: {topol_feat.shape}")
print(f"🔢 Number of NaN values before imputation: {topol_feat.isna().sum().sum()}")
print("\n🗂️ NaN counts per column before imputation:")
print(topol_feat.isna().sum())


📊 Shape of topol_feat: (19168, 24)
🔢 Number of NaN values before imputation: 180022

🗂️ NaN counts per column before imputation:
sum_H_0              8995
sum_H_1              8995
sum_H_2              8995
mean_H_0             8995
mean_H_1             8995
mean_H_2             8995
std_H_0              8995
std_H_1              8995
std_H_2              8995
entropy_H_0          8995
entropy_H_1          9128
entropy_H_2          9530
solv_sum_H_0         3733
solv_sum_H_1         3733
solv_sum_H_2         3733
solv_mean_H_0        3733
solv_mean_H_1        3733
solv_mean_H_2        3733
solv_std_H_0         3733
solv_std_H_1         3733
solv_std_H_2         3733
solv_entropy_H_0     3733
solv_entropy_H_1    16280
solv_entropy_H_2    17804
dtype: int64


In [104]:
# Проверяем форму данных перед импутацией
print(f"\n📊 Shape of topol_feat: {topol_feat.shape}")
print(f"🔢 Number of NaN values before imputation: {topol_feat.isna().sum().sum()}")
print("\n🗂️ NaN counts per column before imputation:")
print(topol_feat.isna().sum())

# Проверяем типы данных и преобразуем все столбцы в числовой формат
print("\n🔍 Checking and converting data types...")
for column in topol_feat.columns:
    # Проверяем исходный тип данных
    original_dtype = topol_feat[column].dtype
    print(f"Column '{column}': original dtype = {original_dtype}")
    
    # Преобразуем в числовой формат, заменяя нечисловые значения на NaN
    topol_feat[column] = pd.to_numeric(topol_feat[column], errors='coerce')
    
    # Проверяем новый тип данных
    new_dtype = topol_feat[column].dtype
    print(f"Column '{column}': new dtype = {new_dtype}, NaN count = {topol_feat[column].isna().sum()}")

# Двухэтапная импутация:
# Шаг 1: Заполнение методом 'nearest' (ближайшее значение)
topol_feat_nearest = topol_feat.copy()


📊 Shape of topol_feat: (19168, 24)
🔢 Number of NaN values before imputation: 180022

🗂️ NaN counts per column before imputation:
sum_H_0              8995
sum_H_1              8995
sum_H_2              8995
mean_H_0             8995
mean_H_1             8995
mean_H_2             8995
std_H_0              8995
std_H_1              8995
std_H_2              8995
entropy_H_0          8995
entropy_H_1          9128
entropy_H_2          9530
solv_sum_H_0         3733
solv_sum_H_1         3733
solv_sum_H_2         3733
solv_mean_H_0        3733
solv_mean_H_1        3733
solv_mean_H_2        3733
solv_std_H_0         3733
solv_std_H_1         3733
solv_std_H_2         3733
solv_entropy_H_0     3733
solv_entropy_H_1    16280
solv_entropy_H_2    17804
dtype: int64

🔍 Checking and converting data types...
Column 'sum_H_0': original dtype = float64
Column 'sum_H_0': new dtype = float64, NaN count = 8995
Column 'sum_H_1': original dtype = float64
Column 'sum_H_1': new dtype = float64, NaN count =

In [114]:
t = pd.read_csv('/Users/egorilin/Desktop/MSU_AI/Topology/Topology_features_metal_complex_with_nan.csv').drop(['Unnamed: 0'],axis=1)
topol_feat_nearest = t.copy()

In [115]:
t.columns

Index(['sum_H_0', 'sum_H_1', 'sum_H_2', 'mean_H_0', 'mean_H_1', 'mean_H_2',
       'std_H_0', 'std_H_1', 'std_H_2', 'entropy_H_0', 'entropy_H_1',
       'entropy_H_2', 'solv_sum_H_0', 'solv_sum_H_1', 'solv_sum_H_2',
       'solv_mean_H_0', 'solv_mean_H_1', 'solv_mean_H_2', 'solv_std_H_0',
       'solv_std_H_1', 'solv_std_H_2', 'solv_entropy_H_0', 'solv_entropy_H_1',
       'solv_entropy_H_2'],
      dtype='object')

In [116]:
topol_feat_nearest = np.array(topol_feat_nearest)
topol_feat_nearest = pd.DataFrame(topol_feat_nearest, columns=t.columns)

In [117]:
for column in tqdm(topol_feat_nearest.columns):
    try:
        # Создаем копию для очистки
        cleaned_column = topol_feat_nearest[column].copy()
        
        # Если столбец содержит строки, очищаем их от проблемных символов
        if cleaned_column.dtype == 'object':
            # Заменяем '/' на '.' (для десятичных дробей) или удаляем
            cleaned_column = cleaned_column.astype(str).str.replace('/', '.', regex=False)
            # Заменяем другие проблемные символы
            cleaned_column = cleaned_column.str.replace(',', '.', regex=False)
            cleaned_column = cleaned_column.str.replace('\\', '', regex=False)
            cleaned_column = cleaned_column.str.replace('|', '', regex=False)
            cleaned_column = cleaned_column.str.replace(' ', '', regex=False)
            # Удаляем нечисловые символы, кроме цифр, точки и знака минус
            cleaned_column = cleaned_column.str.replace(r'[^0-9.-]', '', regex=True)
        
        # Преобразуем в числовой формат
        cleaned_column = pd.to_numeric(cleaned_column, errors='coerce')
        
        # Проверяем, есть ли непустые значения для интерполяции
        if cleaned_column.notna().any():
            # Применяем интерполяцию методом 'nearest'
            cleaned_column = cleaned_column.interpolate(method='nearest')
            
            # Если остались пропуски, заполняем их ближайшими значениями
            if cleaned_column.isna().any():
                cleaned_column = cleaned_column.ffill().bfill()
                
            topol_feat_nearest[column] = cleaned_column
        else:
            # Если нет непустых значений, заполняем нулями
            topol_feat_nearest[column] = 0.0
            
    except Exception as e:
        print(f"Ошибка при обработке колонки '{column}': {e}")
        # В случае ошибки заполняем нулями
        topol_feat_nearest[column] = 0.0
    
    # Гарантируем, что все значения будут float
    topol_feat_nearest[column] = topol_feat_nearest[column].astype(float)
topol_feat_nearest

100%|██████████| 24/24 [00:00<00:00, 450.39it/s]


,sum_H_0,sum_H_1,sum_H_2,mean_H_0,mean_H_1,mean_H_2,std_H_0,std_H_1,std_H_2,entropy_H_0,...,solv_sum_H_2,solv_mean_H_0,solv_mean_H_1,solv_mean_H_2,solv_std_H_0,solv_std_H_1,solv_std_H_2,solv_entropy_H_0,solv_entropy_H_1,solv_entropy_H_2
0,174.767792,18.198271,2.254412,1.266433,0.433292,0.187868,0.206595,0.318165,0.117802,4.914500,...,0.0,1.169003,0.0,0.0,0.179086,0.0,0.0,2.068225,-0.0,-0.0
1,174.767792,18.198271,2.254412,1.266433,0.433292,0.187868,0.206595,0.318165,0.117802,4.914500,...,0.0,1.430349,0.0,0.0,0.341314,0.0,0.0,1.357547,-0.0,-0.0
2,174.767792,18.198271,2.254412,1.266433,0.433292,0.187868,0.206595,0.318165,0.117802,4.914500,...,0.0,1.169003,0.0,0.0,0.179086,0.0,0.0,2.068225,-0.0,-0.0
3,174.767792,18.198271,2.254412,1.266433,0.433292,0.187868,0.206595,0.318165,0.117802,4.914500,...,0.0,1.169003,0.0,0.0,0.179086,0.0,0.0,2.068225,-0.0,-0.0
4,174.767792,18.198271,2.254412,1.266433,0.433292,0.187868,0.206595,0.318165,0.117802,4.914500,...,0.0,1.169003,0.0,0.0,0.179086,0.0,0.0,2.068225,-0.0,-0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19163,76.180099,10.600771,1.611456,1.269668,0.623575,0.268576,0.206686,0.308903,0.099560,4.081655,...,0.0,1.169703,0.0,0.0,0.135189,0.0,0.0,1.603083,-0.0,-0.0
19164,76.180099,10.600771,1.611456,1.269668,0.623575,0.268576,0.206686,0.308903,0.099560,4.081655,...,0.0,1.430349,0.0,0.0,0.341314,0.0,0.0,1.357547,-0.0,-0.0
19165,79.971557,11.428896,1.966793,1.289864,0.634939,0.327799,0.185795,0.253796,0.012683,4.117024,...,0.0,1.430349,0.0,0.0,0.341314,0.0,0.0,1.357547,-0.0,-0.0
19166,79.971557,11.428896,1.966793,1.289864,0.634939,0.327799,0.185795,0.253796,0.012683,4.117024,...,0.0,1.169703,0.0,0.0,0.135189,0.0,0.0,1.603083,-0.0,-0.0


In [119]:
topol_feat_nearest.to_csv('./Topology/Topology_features_metal_complex_imputed.csv')

# Также сохраняем исходный DataFrame с пропусками для отладки
topol_feat.to_csv('./Topology/Topology_features_metal_complex_with_nan.csv')